# LangChain Tools — Complete Tutorial
### *Give your LLM superpowers*

---

> **LangChain:** 1.3.6 | **Python:** 3.11.9  
> **Prerequisite:** Chains tutorial


*No extra API keys needed for most tools — DuckDuckGo, Wikipedia, Python REPL are all free.*

---
## Setup

In [ ]:
!pip install -q langchain langchain-openai langchain-community openai

In [1]:
!pip install -q duckduckgo-search wikipedia requests


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import importlib.metadata

# List the distribution package names
packages = ["langchain", "langchain-community", "langchain-openai", "openai",
            "langchain_core", "duckduckgo-search",]

for package in packages:
    try:
        version = importlib.metadata.version(package)
        print(f"{package} version: {version}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{package} is not installed in this environment.")

langchain version: 1.3.6
langchain-community version: 0.4.2
langchain-openai version: 1.2.2
openai version: 2.36.0
langchain_core version: 1.4.1
duckduckgo-search version: 8.1.1


In [7]:
import os
from dotenv import load_dotenv

load_dotenv() # read .env file

# TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

In [8]:
import os
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(api_key = OPENAI_API_KEY,
                 model="gpt-4o-mini", 
                 temperature=0
)  # temperature=0 for tool use

response = llm.invoke("Confirm connection by replying with the word 'Active'")
    
print("\nConnection Successful!")
print(f"AI Response: '{response.content.strip()}'")


Connection Successful!
AI Response: 'Active'


---
## 1️) Why Do We Need Tools?

### LLMs Are Smart But Limited

| LLM Can ✅ | LLM Cannot ❌ |
|-----------|-------------|
| Reason, explain, summarise | Know today's news |
| Write, translate, code | Do precise arithmetic |
| Answer questions from training data | Access the internet |
| Understand context | Check current stock prices |

### The Gap

```python
llm.invoke('What is 2847 * 3921?')
# LLM might say: 11,162,487  -- WRONG! Correct: 11,155,287

llm.invoke('What is today\'s date?')
# LLM: 'I don\'t have access to real-time information...'

llm.invoke('What is the temperature in New Delhi ?')
# LLM: 'I don\'t have access to real-time information...'
```

**Tools** bridge this gap — they let the LLM call real code and real APIs.

### How Tools Work

```
User: 'What is 2847 * 3921?'
         ↓
LLM sees the question + available tools
         ↓
LLM decides: 'I should use the calculator tool'
         ↓
Tool runs: calculator('2847 * 3921') -> 11,155,287
         ↓
LLM forms response: '2847 x 3921 = 11,155,287'
```

In [9]:
# Demonstrate the problem — LLMs are unreliable at arithmetic

response = llm.invoke('What is 2847 * 3921? Give just the number, no explanation.')
print('LLM answer:   ', response.content)
print('Correct answer:', 2847 * 3921)
print('Match?', str(2847 * 3921) in response.content.replace(',',''))

LLM answer:    11,155,787
Correct answer: 11163087
Match? False


---
## 2️ Custom Tools — The `@tool` Decorator

The simplest way to create a tool: any Python function with `@tool`.

### Three things the LLM uses:
1. **Function name** — identifies the tool
2. **Docstring** — LLM reads this to decide WHEN to use it
3. **Type hints** — tells LangChain input/output types

```python
@tool
def my_tool(input: str) -> str:
    """Describe WHAT this does and WHEN to use it.
    The LLM reads this like instructions."""
    return "result"
```

In [10]:
from langchain_core.tools import tool

# Simplest possible tool
@tool
def greet(name: str) -> str:
    """
    Greet a person by name. Use when you need to say hello to someone.
    """
    return f"Hello, {name}! Welcome to LangChain Tools! 👋"

print(greet.invoke({"name": "Priya"}))

print("--- Tool Metadata ---")
print("Name:       ", greet.name)
print("Description:", greet.description)
print("Args schema:", greet.args)

Hello, Priya! Welcome to LangChain Tools! 👋
--- Tool Metadata ---
Name:        greet
Description: Greet a person by name. Use when you need to say hello to someone.
Args schema: {'name': {'title': 'Name', 'type': 'string'}}


In [11]:
# Tool with multiple parameters

@tool
def calculate_bmi(weight_kg: float, height_m: float) -> str:
    """
    Calculate Body Mass Index (BMI) given weight in kg and height in meters.
    Use when someone asks about BMI or healthy weight range.
    """
    bmi = weight_kg / (height_m ** 2)
    if bmi < 18.5:   category = 'Underweight'
    elif bmi < 25:   category = 'Normal weight'
    elif bmi < 30:   category = 'Overweight'
    else:            category = 'Obese'
    return f"BMI: {bmi:.1f} — {category}"

print(calculate_bmi.invoke({"weight_kg": 70, "height_m": 1.75}))
print(calculate_bmi.invoke({"weight_kg": 50, "height_m": 1.70}))

BMI: 22.9 — Normal weight
BMI: 17.3 — Underweight


In [12]:
from typing import Optional

# Tool with optional parameters
@tool
def format_currency(amount: float, currency: str = 'USD', decimals: int = 2) -> str:
    """
    Format a number as currency. Defaults to USD with 2 decimal places.
    Supports USD, EUR, GBP, INR, JPY. Use for any money formatting request.
    """
    symbols = {'USD': '$', 'EUR': '€', 'GBP': '£', 'INR': '₹', 'JPY': '¥'}
    symbol = symbols.get(currency.upper(), currency + ' ')
    return f"{symbol}{amount:,.{decimals}f}"

print(format_currency.invoke({"amount": 1234567.89}))
print(format_currency.invoke({"amount": 9999.5, "currency": "INR"}))
print(format_currency.invoke({"amount": 500, "currency": "EUR", "decimals": 0}))

$1,234,567.89
₹9,999.50
€500


In [13]:
# Best practice demo: vague vs descriptive docstrings

@tool
def bad_tool(x: str) -> str:
    """
    Does stuff with input.
    """
    return x

@tool
def word_counter(text: str) -> str:
    """
    Count the number of words in a piece of text.
    Use when asked 'how many words', 'word count', or the length of a text in words.
    Input should be the full text string.
    """
    count = len(text.split())
    return f"The text contains {count} words."

print("Bad  tool:", bad_tool.description)
print("Good tool:", word_counter.description)

Bad  tool: Does stuff with input.
Good tool: Count the number of words in a piece of text.
Use when asked 'how many words', 'word count', or the length of a text in words.
Input should be the full text string.


---
## 3️. Math & Data Tools

LLMs are notoriously unreliable at arithmetic. Always use a tool for calculations.

| Tool | Use case |
|------|----------|
| Calculator | Safe math expression evaluation |
| Stats calculator | Mean, median, std dev etc. |
| Python REPL | Run arbitrary Python |
| Unit converter | Length, weight, temperature, speed |

In [14]:
import math

@tool
def calculator(expression: str) -> str:
    """
    Evaluate a mathematical expression. Input must be valid Python math.
    Supports: +, -, *, /, **, sqrt, sin, cos, log, pi, e.
    Examples: '2847 * 3921', 'math.sqrt(144)', 'math.pi * 5**2'
    ALWAYS use for any calculation instead of computing mentally.
    """
    try:
        allowed = {k: getattr(math, k) for k in dir(math) if not k.startswith('_')}
        allowed['abs'] = abs
        allowed['round'] = round
        result = eval(expression, {"__builtins__": {}}, allowed)
        return f"{expression} = {result}"
    except Exception as e:
        return f"Error evaluating '{expression}': {e}"

print(calculator.invoke({"expression": "2847 * 3921"}))
print(calculator.invoke({"expression": "math.sqrt(144)"}))
print(calculator.invoke({"expression": "math.pi * 7**2"}))
print(calculator.invoke({"expression": "math.log(1000, 10)"}))
print(calculator.invoke({"expression": "(15 + 27 + 33 + 42) / 4"}))

2847 * 3921 = 11163087
Error evaluating 'math.sqrt(144)': name 'math' is not defined
Error evaluating 'math.pi * 7**2': name 'math' is not defined
Error evaluating 'math.log(1000, 10)': name 'math' is not defined
(15 + 27 + 33 + 42) / 4 = 29.25


In [15]:
import statistics

@tool
def stats_calculator(numbers: str) -> str:
    """
    Calculate statistics for a list of numbers.
    Input: comma-separated numbers like '10, 20, 30, 40, 50'
    Returns: mean, median, std deviation, min, max, range.
    """
    try:
        nums = [float(x.strip()) for x in numbers.split(',')]
        result = {
            'count':   len(nums),
            'mean':    round(statistics.mean(nums), 4),
            'median':  statistics.median(nums),
            'std_dev': round(statistics.stdev(nums), 4) if len(nums) > 1 else 0,
            'min':     min(nums),
            'max':     max(nums),
            'range':   max(nums) - min(nums)
        }
        return "\n".join(f"{k}: {v}" for k, v in result.items())
    except Exception as e:
        return f"Error: {e}"

scores = "72, 85, 91, 68, 77, 95, 83, 79, 88, 64"
print("Exam Score Statistics:")
print(stats_calculator.invoke({"numbers": scores}))

Exam Score Statistics:
count: 10
mean: 80.2
median: 81.0
std_dev: 10.0973
min: 64.0
max: 95.0
range: 31.0


In [17]:
# Python REPL Tool — run actual Python code
# Most powerful data tool — great for complex data manipulation

import sys
from io import StringIO

# Clean replacement — no extra packages needed
import sys
from io import StringIO
from langchain_core.tools import tool

@tool
def python_repl(code: str) -> str:
    """
    Execute Python code and return the printed output.
    Use for data manipulation, calculations, sorting, or any Python logic.
    Input should be valid Python code as a string.
    """
    old_stdout = sys.stdout
    sys.stdout = StringIO()
    try:
        exec(code, {})
        output = sys.stdout.getvalue()
    except Exception as e:
        output = f"Error: {e}"
    finally:
        sys.stdout = old_stdout
    return output if output else "Code ran successfully (no output)"

# Same test — works identically
result = python_repl.invoke({"code": """
students = [
    {"name": "Alice", "score": 85},
    {"name": "Bob",   "score": 72},
    {"name": "Carol", "score": 91},
    {"name": "Dave",  "score": 68},
    {"name": "Eve",   "score": 95},
]
ranked = sorted(students, key=lambda x: x['score'], reverse=True)
print("=== CLASS RANKING ===")
for i, s in enumerate(ranked, 1):
    grade = "A" if s["score"] >= 90 else "B" if s["score"] >= 80 else "C"
    print(f"{i}. {s['name']:10s} {s['score']}  Grade: {grade}")
avg = sum(s['score'] for s in students) / len(students)
print(f"\\nClass average: {avg:.1f}")
"""})

print(result)

=== CLASS RANKING ===
1. Eve        95  Grade: A
2. Carol      91  Grade: A
3. Alice      85  Grade: B
4. Bob        72  Grade: C
5. Dave       68  Grade: C

Class average: 82.2



In [18]:
@tool
def unit_converter(value: float, from_unit: str, to_unit: str) -> str:
    """
    Convert between common units of measurement.
    Supports length (km/miles/meters/feet/cm/inches),
    weight (kg/pounds/grams), temperature (celsius/fahrenheit/kelvin),
    volume (liters/gallons/ml), speed (kmh/mph).
    Example: unit_converter(100, 'km', 'miles')
    """
    conversions = {
        ('km','miles'):          lambda x: x * 0.621371,
        ('miles','km'):          lambda x: x * 1.60934,
        ('meters','feet'):       lambda x: x * 3.28084,
        ('feet','meters'):       lambda x: x * 0.3048,
        ('cm','inches'):         lambda x: x * 0.393701,
        ('inches','cm'):         lambda x: x * 2.54,
        ('kg','pounds'):         lambda x: x * 2.20462,
        ('pounds','kg'):         lambda x: x * 0.453592,
        ('celsius','fahrenheit'):lambda x: x * 9/5 + 32,
        ('fahrenheit','celsius'):lambda x: (x - 32) * 5/9,
        ('celsius','kelvin'):    lambda x: x + 273.15,
        ('kelvin','celsius'):    lambda x: x - 273.15,
        ('liters','gallons'):    lambda x: x * 0.264172,
        ('gallons','liters'):    lambda x: x * 3.78541,
        ('kmh','mph'):           lambda x: x * 0.621371,
        ('mph','kmh'):           lambda x: x * 1.60934,
    }
    key = (from_unit.lower(), to_unit.lower())
    if key in conversions:
        result = conversions[key](value)
        return f"{value} {from_unit} = {result:.4f} {to_unit}"
    return f"Conversion from {from_unit!r} to {to_unit!r} not supported."

print(unit_converter.invoke({"value": 100,  "from_unit": "km",      "to_unit": "miles"}))
print(unit_converter.invoke({"value": 37,   "from_unit": "celsius", "to_unit": "fahrenheit"}))
print(unit_converter.invoke({"value": 70,   "from_unit": "kg",      "to_unit": "pounds"}))
print(unit_converter.invoke({"value": 120,  "from_unit": "kmh",     "to_unit": "mph"}))

100.0 km = 62.1371 miles
37.0 celsius = 98.6000 fahrenheit
70.0 kg = 154.3234 pounds
120.0 kmh = 74.5645 mph


---
## 4️. Web & Search Tools

The most important category — these give LLMs access to **current information**.
Your LLM training data has a cutoff date. Search tools fix that.

| Tool | Free? | Best for |
|------|-------|----------|
| DuckDuckGo Search | ✅ Free | General web search |
| Wikipedia | ✅ Free | Encyclopedic knowledge |
| Web Fetch (requests) | ✅ Free | Read any webpage |
| Tavily Search | API key | High quality search results |

In [19]:
from langchain_community.tools import DuckDuckGoSearchRun, DuckDuckGoSearchResults

# Option A: plain text summary — easiest to use
search = DuckDuckGoSearchRun()

result = search.invoke("latest developments in quantum computing 2025")

print("=== DuckDuckGoSearchRun (plain text) ===")
print(result[:600])

=== DuckDuckGoSearchRun (plain text) ===
August 19, 2025 - Insights from the “Quantum Index Report 2025” include the following: Quantum processor performance is improving, with the U. S. leading the field. Two-dozen manufacturers are now commercially offering more than 40 quantum processing units ... November 19, 2025 - But as the quantum computers got bigger, the error rates would go up even faster, meaning that it was impossible to catch up. In the last couple of years, researchers started to find ways around this problem, both in making the individual qubits less prone to errors, and by improving the error-correction mechanisms. T


In [20]:
# Option B: structured results with title + URL + snippet
search_results = DuckDuckGoSearchResults(num_results=3)

results = search_results.invoke("Python programming news 2025")

print("=== DuckDuckGoSearchResults (structured) ===")
print(results[:800])

=== DuckDuckGoSearchResults (structured) ===
snippet: November 4, 2025 - The Python Language Summit of 2025 revealed that “Somewhere between one-quarter and one-third of all native code being uploaded to PyPI for new projects uses Rust”, indicating that “people are choosing to start new projects using Rust”. ..., title: The State of Python 2025: Trends and Survey Insights - The JetBrains Blog, link: https://blog.jetbrains.com/pycharm/2025/08/the-state-of-python-2025/, snippet: 2 weeks ago - Discover the latest Python development news, repositories, and conferences at Hackertab., title: Stay Ahead with Python News – Updated Every Hour, link: https://hackertab.dev/topics/python, snippet: December 8, 2025 - PEP 810 brings lazy imports to Python 3.15, PyPI tightens 2FA security, and Django 6.0 reaches release candidate. Catch up on all t


In [25]:
from langchain_tavily import TavilySearch

tavily_search = TavilySearch()

extracted_info = tavily_search.run("What is recent development Agentic AI?")

extracted_info

{'query': 'What is recent development Agentic AI?',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://www.ibm.com/think/insights/agentic-ai',
   'title': "Agentic AI: 4 reasons why it's the next big thing in AI research - IBM",
   'content': 'Agentic AI refers to a system or program that is capable of autonomously performing tasks on behalf of a user or another system by designing its workflow and',
   'score': 0.7304634,
   'raw_content': None},
  {'url': 'https://www.rishabhsoft.com/blog/agentic-ai-in-software-development',
   'title': 'Agentic AI in Software Development Accelerating Modern Delivery',
   'content': "From multi-agent systems to self-improving solutions, Agentic AI marks the beginning of a new game in software development. Let's explore",
   'score': 0.5769478,
   'raw_content': None},
  {'url': 'https://about.gitlab.com/the-source/ai/emerging-agentic-ai-trends-reshaping-software-development',
   'title': 'Emerging agentic AI t

In [21]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

wiki = WikipediaQueryRun(
    api_wrapper=WikipediaAPIWrapper(
        top_k_results=1,
        doc_content_chars_max=1000
    )
)

result = wiki.invoke("Large Language Models history")
print("=== Wikipedia Result ===")
print(result)

=== Wikipedia Result ===
Page: Large language model
Summary: A large language model (LLM) is a neural network trained on a vast amount of text for natural language processing tasks, especially language generation. LLMs can typically generate, summarize, translate, and analyze text in many contexts, and are a foundational technology behind modern chatbots. Biased or inaccurate training data can make an LLM's output less reliable.
LLMs are typically based on transformer architecture. Generative pre-trained transformers (GPTs) are a type of LLM that is pre-trained to predict the next word. GPTs are then often fine-tuned to follow instructions and to behave as assistants.
Benchmark evaluations for LLMs attempt to measure model reasoning, factual accuracy, alignment, and safety.


In [23]:
import requests, re

@tool
def fetch_webpage(url: str) -> str:
    """
    Fetch and return the text content of any webpage URL.
    Use when you need to read the content of a specific website.
    Input should be a full URL starting with https://
    """
    try:
        headers = {"User-Agent": "Mozilla/5.0"}
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        text = re.sub(r'<[^>]+>', ' ', response.text)
        text = re.sub(r'\s+', ' ', text).strip()
        return f"Content from {url}:\n{text[:1500]}..."
    except Exception as e:
        return f"Could not fetch {url}: {e}"

# result = fetch_webpage.invoke({"url": "https://httpbin.org/json"})
result = fetch_webpage.invoke({"url": "https://loxfordacademy.com/blog/uncategorized/hugging-face-the-github-of-ai-models/"})

print(result[:1000])

Content from https://loxfordacademy.com/blog/uncategorized/hugging-face-the-github-of-ai-models/:
/* */ /* */ :root { --lp-container-max-width: 1290px; --lp-cotainer-padding: 1rem; --lp-primary-color: #ffb606; --lp-secondary-color: #442e66; } Hugging Face: The GitHub of AI Models &#8211; loxfordacademy.com img:is([sizes=auto i],[sizes^="auto," i]){contain-intrinsic-size:3000px 1500px} /*# sourceUR


# LOOK AT WEBSITE